# Benchmark multibin $\Lambda$CDM AP MCMC warmup

This notebook benchmarks the multibin AP likelihood used in
`example/mcmc/mcmc_multibin_LCDM_AP.ipynb`, with emphasis on

- theory, log-posterior, and gradient cost,
- NUTS warmup cost for different mass-matrix strategies,
- the effect of tabulating $\chi(z)$ once per cosmology and interpolating
  $D_A(z)$ at the survey-bin redshifts.

The working hypothesis is simple: if `fixed Fisher diagonal` and `adapted Fisher diagonal`
finish in similar time, then the bottleneck is not the adaptation bookkeeping itself but the
repeated gradient evaluations of the likelihood during warmup.

## Modules

In [ ]:
import time
from functools import partial

import matplotlib
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": "Computer Modern",
    "font.size": 18,
})

import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.linalg import block_diag, inv

from ps_1loop_jax import background as bg

from jaxptpolypol.model import CosmoEmulator, PS1LoopModel
from jaxptpolypol.params import CosmoParams, SurveyParams, pack_multibin_params
from jaxptpolypol.covariance import gaussian_covariance
from jaxptpolypol.inference import fisher_matrix, marginalize_fisher, fixed_and_varied_indices
from jaxptpolypol.theory import make_pk_ell_fn, compute_fiducial_distances
from jaxptpolypol.sampler import (
    make_transform,
    make_full_params_fn,
    make_gaussian_log_prior,
    make_log_posterior,
    warmup_nuts,
)

## Benchmark configuration

In [ ]:
LAPTOP_MODE = True
BACKGROUND_NZ = 256
MICRO_REPEATS = 3
WARMUP_REPEATS = 1

if LAPTOP_MODE:
    N_K = 12
    N_GL = 16
    NUM_WARMUP_BENCH = 50
    NUM_CHAINS_BENCH = 1
else:
    N_K = 20
    N_GL = 16
    NUM_WARMUP_BENCH = 150
    NUM_CHAINS_BENCH = 1

print(f"Mode: {'LAPTOP' if LAPTOP_MODE else 'SERVER'}")
print(f"  N_K = {N_K}, N_GL = {N_GL}, background_nz = {BACKGROUND_NZ}")
print(f"  micro repeats = {MICRO_REPEATS}, warmup repeats = {WARMUP_REPEATS}")
print(f"  warmup steps per benchmark = {NUM_WARMUP_BENCH}")

## Problem setup helpers

In [ ]:
MODEL_PATH = '/Users/nguyenmn/cosmopower-jax-for-pfs/cosmology/jense2024/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_Pk_lin.npz'
ELLS = (0, 2, 4)
Z_BINS = (0.7, 0.9, 1.1, 1.3, 1.5, 1.8, 2.2)
V_BINS = tuple(v * 1000.**3 for v in (0.59, 0.79, 0.96, 1.09, 1.19, 2.58, 2.71))
KNL_BINS = (0.52, 0.65, 0.82, 1.02, 1.29, 1.82, 2.88)
N_BINS = (3.06e-4, 9.61e-4, 9.75e-4, 6.54e-4, 3.40e-4, 2.02e-4, 3.51e-4)


def b1z(z): return 0.9 + 0.4 * z

def b2z(z): return -0.704 - 0.208 * z + 0.183 * z**2 - 0.00771 * z**3

def bG2z(z): return -(2.0 / 7.0) * (b1z(z) - 1.0)

def bGamma3z(z): return (23.0 / 42.0) * (b1z(z) - 1.0)


def make_fiducial_cosmo():
    return CosmoParams({
        'ombh2': 0.02242,
        'omch2': 0.11933,
        'logA': 3.047,
        'ns': 0.9665,
        'h': 0.6766,
        'z': 0.7,
        'A_b': 3.13,
        'eta_b': 0.603,
        'logT_AGN': 7.8,
    })


def Dplusz(cosmo_dict, z):
    return float(bg.growth_factor(
        cosmo_dict['ombh2'], cosmo_dict['omch2'], cosmo_dict['h'], z, mnu=0.06
    ))


def c0z(cosmo_dict, z): return 0.35 / Dplusz(cosmo_dict, z)**2

def c2z(cosmo_dict, z): return 0.15 / Dplusz(cosmo_dict, z)**2

def c4z(cosmo_dict, z): return 0.05 / Dplusz(cosmo_dict, z)**2


def make_surveys(cosmo_dict):
    surveys = []
    for z, knl, nd in zip(Z_BINS, KNL_BINS, N_BINS):
        surveys.append(SurveyParams({
            'bias': {'b1': b1z(z), 'b2': b2z(z), 'bG2': bG2z(z), 'bGamma3': bGamma3z(z)},
            'ctr': {'c0': c0z(cosmo_dict, z), 'c2': c2z(cosmo_dict, z), 'c4': c4z(cosmo_dict, z), 'cfog': knl**(-4)},
            'stoch': {'P_shot': 1.0, 'a0': 0.0, 'a2': 0.0},
            'k_nl': knl,
            'ndens': nd,
        }))
    return surveys


def block_ready(x):
    jax.block_until_ready(x)
    return x


def benchmark_callable(fn, repeats=3, block=block_ready):
    times = []
    last = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        last = fn()
        block(last)
        times.append(time.perf_counter() - t0)
    return {
        'mean_s': float(np.mean(times)),
        'std_s': float(np.std(times)),
        'times_s': times,
        'last': last,
    }

## Build direct and tabulated AP problems

In [ ]:
pklin_emulator = CosmoEmulator(probe='custom_log', emulator_path=MODEL_PATH)
ps1loop_model = PS1LoopModel(do_irres=True)


def build_problem(background_mode):
    cosmo = make_fiducial_cosmo()
    surveys = make_surveys(cosmo.to_dict())
    packed_params = pack_multibin_params(cosmo, surveys)

    n_cosmo = sum(cosmo.param_sizes)
    n_survey = len(surveys[0].param_keys)
    n_bins = len(Z_BINS)
    n_ell = len(ELLS)
    k = jnp.linspace(5e-3, 0.25, N_K)

    Hz_fid, DAz_fid = compute_fiducial_distances(cosmo, Z_BINS)
    pk_fn = make_pk_ell_fn(
        ells=ELLS,
        pklin_emulator=pklin_emulator,
        ps1loop_model=ps1loop_model,
        cosmo_keys=cosmo.param_keys,
        cosmo_sizes=cosmo.param_sizes,
        survey_keys=surveys[0].param_keys,
        ap=True,
        z_bins=Z_BINS,
        Hz_fid=Hz_fid,
        DAz_fid=DAz_fid,
        n_gl=N_GL,
        background_mode=background_mode,
        background_nz=BACKGROUND_NZ,
    )
    jitted_pk = jax.jit(pk_fn)
    theory_fid = jitted_pk(packed_params, k=k)
    theory_fid.block_until_ready()

    pk_all = theory_fid.reshape(n_bins, n_ell, len(k))
    shot_noise = jnp.zeros_like(pk_all)
    for b in range(n_bins):
        shot_noise = shot_noise.at[b, 0, :].set(1.0 / N_BINS[b])
    data = (pk_all + shot_noise).reshape(-1)

    dk = k[1] - k[0]
    cov = block_diag(*[
        gaussian_covariance(V_BINS[b], k, dk, *(pk_all[b] + shot_noise[b]))
        for b in range(n_bins)
    ])
    cov_inv = inv(cov)

    jac = jax.jacfwd(jitted_pk, argnums=0)(packed_params, k=k)
    F_full = fisher_matrix(cov, jac)

    fixed_cosmo_idx = [5, 6, 7, 8]
    fixed_survey_offsets = [11, 12]
    _, varied_idx = fixed_and_varied_indices(
        n_cosmo, n_survey, n_bins, fixed_cosmo_idx, fixed_survey_offsets
    )
    F_varied = marginalize_fisher(F_full, varied_idx)
    F_varied_inv = inv(F_varied)
    fisher_sigma = jnp.sqrt(jnp.diag(F_varied_inv))
    fid_varied = packed_params[jnp.array(varied_idx)]

    to_whitened, to_physical = make_transform(center=fid_varied, scale=fisher_sigma)
    full_params_fn = make_full_params_fn(packed_params, varied_idx)
    cosmo_in_varied = [varied_idx.index(i) for i in [0, 1, 2, 3, 4]]
    prior_entries = [
        (cosmo_in_varied[0], 0.02218, 0.00055),
        (cosmo_in_varied[3], 0.9649, 0.042),
    ]
    log_prior = make_gaussian_log_prior(len(varied_idx), prior_entries)
    theory_fn = partial(jitted_pk, k=k)
    log_post = make_log_posterior(
        theory_fn=theory_fn,
        data=data,
        cov_inv=cov_inv,
        log_prior_fn=log_prior,
        to_physical=to_physical,
        full_params_fn=full_params_fn,
    )
    grad_log_post = jax.jit(jax.grad(log_post))

    x0 = jnp.zeros(len(varied_idx))
    log_post(x0).block_until_ready()
    grad_log_post(x0).block_until_ready()

    S = jnp.diag(fisher_sigma)
    F_inv_whitened = S @ F_varied_inv @ S

    return {
        'background_mode': background_mode,
        'packed_params': packed_params,
        'k': k,
        'jitted_pk': jitted_pk,
        'theory_fid': theory_fid,
        'log_post': log_post,
        'grad_log_post': grad_log_post,
        'x0': x0,
        'n_varied': len(varied_idx),
        'F_inv_whitened': F_inv_whitened,
        'F_inv_whitened_diag': jnp.diag(F_inv_whitened),
    }


problem_direct = build_problem('direct')
problem_tabulated = build_problem('tabulated')
print(f"Built direct and tabulated problems with n_varied = {problem_tabulated['n_varied']}")

## Microbenchmarks: theory and likelihood path

In [ ]:
theory_diff = problem_tabulated['theory_fid'] - problem_direct['theory_fid']
rel_theory_diff = jnp.max(
    jnp.abs(theory_diff) / jnp.maximum(1.0, jnp.abs(problem_direct['theory_fid']))
)
print(f"Max relative theory difference (tabulated vs direct): {float(rel_theory_diff):.3e}")

micro_results = {}
for label, problem in [('direct', problem_direct), ('tabulated', problem_tabulated)]:
    micro_results[(label, 'theory')] = benchmark_callable(
        lambda: problem['jitted_pk'](problem['packed_params'], k=problem['k']),
        repeats=MICRO_REPEATS,
    )
    micro_results[(label, 'log_post')] = benchmark_callable(
        lambda: problem['log_post'](problem['x0']),
        repeats=MICRO_REPEATS,
    )
    micro_results[(label, 'grad_log_post')] = benchmark_callable(
        lambda: problem['grad_log_post'](problem['x0']),
        repeats=MICRO_REPEATS,
    )

print(f"{'Path':<20s} {'Mode':<12s} {'Mean [s]':>12s} {'Std [s]':>12s}")
print('-' * 60)
for metric in ['theory', 'log_post', 'grad_log_post']:
    for label in ['direct', 'tabulated']:
        result = micro_results[(label, metric)]
        print(f"{metric:<20s} {label:<12s} {result['mean_s']:12.4f} {result['std_s']:12.4f}")

## Warmup benchmark: mass-matrix strategies

In [ ]:
warmup_configs = [
    (
        'fixed Fisher diag',
        dict(
            adapt_mass_matrix=False,
            mass_matrix_type='diagonal',
            initial_inverse_mass_matrix=problem_tabulated['F_inv_whitened_diag'],
        ),
    ),
    (
        'adapt Fisher diag',
        dict(
            adapt_mass_matrix=True,
            mass_matrix_type='diagonal',
            initial_inverse_mass_matrix=problem_tabulated['F_inv_whitened_diag'],
        ),
    ),
    (
        'adapt identity diag',
        dict(
            adapt_mass_matrix=True,
            mass_matrix_type='diagonal',
            initial_inverse_mass_matrix=None,
        ),
    ),
    (
        'adapt Fisher dense',
        dict(
            adapt_mass_matrix=True,
            mass_matrix_type='dense',
            initial_inverse_mass_matrix=problem_tabulated['F_inv_whitened'],
        ),
    ),
]

warmup_results = []
for name, kwargs in warmup_configs:
    times = []
    last_params = None
    for rep in range(WARMUP_REPEATS):
        t0 = time.perf_counter()
        _, last_params = warmup_nuts(
            jax.random.key(1234 + rep),
            problem_tabulated['log_post'],
            problem_tabulated['x0'],
            num_warmup=NUM_WARMUP_BENCH,
            num_chains=NUM_CHAINS_BENCH,
            **kwargs,
        )
        jax.block_until_ready(last_params['step_size'])
        jax.block_until_ready(last_params['inverse_mass_matrix'])
        times.append(time.perf_counter() - t0)

    warmup_results.append({
        'name': name,
        'mean_s': float(np.mean(times)),
        'std_s': float(np.std(times)),
        'step_size': float(last_params['step_size'][0]),
    })

print(f"{'Configuration':<22s} {'Mean [s]':>12s} {'Std [s]':>12s} {'Step size':>12s}")
print('-' * 64)
for row in warmup_results:
    print(f"{row['name']:<22s} {row['mean_s']:12.4f} {row['std_s']:12.4f} {row['step_size']:12.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.bar([row['name'] for row in warmup_results], [row['mean_s'] for row in warmup_results],
       color=['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728'])
ax.set_ylabel('Warmup time [s]')
ax.set_title(f'NUTS warmup benchmark ({NUM_WARMUP_BENCH} steps, {NUM_CHAINS_BENCH} chain)')
ax.tick_params(axis='x', rotation=20)
plt.show()

## Interpretation guide

Use the benchmark output as follows:

- If `tabulated` is materially faster than `direct` for `theory`, `log_post`, or
  `grad_log_post`, then repeated AP distance evaluation is a real bottleneck.
- If `fixed Fisher diag` and `adapt Fisher diag` have similar warmup times, then
  the expensive part of warmup is the repeated NUTS gradient work, not the diagonal
  mass-matrix update itself.
- If `adapt Fisher dense` is much slower, the dense inverse mass matrix is likely
  not worth its warmup cost for this problem unless it dramatically reduces
  divergences or integration steps.
- If `adapt identity diag` is slower than `adapt Fisher diag`, Fisher seeding is
  still helping even after whitening.